## Comparative Performance of Active Learning Strategies

In [2]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    try:
        tf.config.experimental.set_memory_growth(physical_devices[0], True)
        print("✅ GPU memory growth enabled.")
    except Exception as e:
        print(f"⚠️ Could not set GPU memory growth: {e}")
else:
    print("❌ No GPU available.")


# Load MNIST dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0
x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

# Define a simple CNN model
def create_cnn_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
        tf.keras.layers.MaxPooling2D(2,2),
        tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
        tf.keras.layers.MaxPooling2D(2,2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')  
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Settings
np.random.seed(42)
initial_labeled_samples = 1000
query_batch_size = 500
num_iterations = 20
epochs_per_round = 10

# Initialize results
approaches = ['Random Sampling', 'Uncertainty Sampling', 'Bayesian Active Learning', 'Hybrid Query Strategy']
accuracies = {approach: [] for approach in approaches}

losses = {approach: [] for approach in approaches}
l2_norms = {approach: [] for approach in approaches}
query_entropies = {a: [] for a in approaches}
query_diversity = {a: [] for a in approaches}


# Active Learning Strategies
def random_sampling(x_unlabeled, num_samples):
    return np.random.choice(len(x_unlabeled), num_samples, replace=False)

def uncertainty_sampling(model, x_unlabeled, num_samples):
    predictions = model.predict(x_unlabeled)
    uncertainty_scores = -np.sum(predictions * np.log(predictions + 1e-10), axis=1)
    return np.argsort(uncertainty_scores)[-num_samples:]

def bayesian_active_learning(model, x_unlabeled, num_samples, num_mcdropout=10):
    mcd_predictions = np.array([model.predict(x_unlabeled) for _ in range(num_mcdropout)])
    mean_prediction = np.mean(mcd_predictions, axis=0)
    uncertainty_scores = -np.sum(mean_prediction * np.log(mean_prediction + 1e-10), axis=1)
    return np.argsort(uncertainty_scores)[-num_samples:]

def hybrid_query_strategy(model, x_labeled, x_unlabeled, num_samples):
    predictions = model.predict(x_unlabeled)
    uncertainty_scores = -np.sum(predictions * np.log(predictions + 1e-10), axis=1)
    diversity_scores = np.linalg.norm(x_unlabeled.reshape(len(x_unlabeled), -1) - np.mean(x_labeled.reshape(len(x_labeled), -1), axis=0), axis=1)
    hybrid_scores = 0.5 * uncertainty_scores + 0.5 * diversity_scores
    return np.argsort(hybrid_scores)[-num_samples:]

def compute_l2_norm(model):
    l2_total = 0.0
    for layer in model.trainable_weights:
        l2_total += tf.reduce_sum(tf.square(layer)).numpy()
    return np.sqrt(l2_total)

def select_balanced_per_class(scores, y_unlabeled, k_per_class=50, num_classes=10):
    selected_indices = []
    for cls in range(num_classes):
        cls_indices = np.where(y_unlabeled == cls)[0]
        if len(cls_indices) == 0:
            continue
        cls_scores = scores[cls_indices]
        top_k = min(k_per_class, len(cls_indices))
        selected = cls_indices[np.argsort(cls_scores)[-top_k:]]
        selected_indices.extend(selected)
    return np.array(selected_indices)

for approach in approaches:
    print(f"\n🚀 Starting approach: {approach}")

    labeled_indices = np.random.choice(len(x_train), initial_labeled_samples, replace=False)
    unlabeled_indices = np.setdiff1d(np.arange(len(x_train)), labeled_indices)
    x_labeled, y_labeled = x_train[labeled_indices], y_train[labeled_indices]
    x_unlabeled, y_unlabeled = x_train[unlabeled_indices], y_train[unlabeled_indices]

    model = create_cnn_model()
    model.fit(x_labeled, y_labeled, epochs=epochs_per_round, verbose=0)

    for i in range(num_iterations):
        print(f"\n🔁 Iteration {i+1}")

        if approach == 'Random Sampling':
            scores = np.random.rand(len(x_unlabeled))
        elif approach == 'Uncertainty Sampling':
            preds = model.predict(x_unlabeled, verbose=0)
            scores = -np.sum(preds * np.log(preds + 1e-10), axis=1)
        elif approach == 'Bayesian Active Learning':
            mcd_preds = np.array([model.predict(x_unlabeled, verbose=0) for _ in range(10)])
            mean_preds = np.mean(mcd_preds, axis=0)
            scores = -np.sum(mean_preds * np.log(mean_preds + 1e-10), axis=1)
        elif approach == 'Hybrid Query Strategy':
            preds = model.predict(x_unlabeled, verbose=0)
            uncertainty_scores = -np.sum(preds * np.log(preds + 1e-10), axis=1)
            x_q = x_unlabeled.reshape(len(x_unlabeled), -1)
            x_l = x_labeled.reshape(len(x_labeled), -1)
            diversity_scores = np.linalg.norm(x_q - np.mean(x_l, axis=0), axis=1)
            scores = 0.5 * uncertainty_scores + 0.5 * diversity_scores

        selected_indices = select_balanced_per_class(scores, y_unlabeled)

        entropies = scores[selected_indices]
        query_entropies[approach].append(np.mean(entropies))

        x_q = x_unlabeled[selected_indices].reshape(len(selected_indices), -1)
        x_l = x_labeled.reshape(len(x_labeled), -1)
        avg_dist = np.mean([np.linalg.norm(xq - np.mean(x_l, axis=0)) for xq in x_q])
        query_diversity[approach].append(avg_dist)

        new_labeled_indices = unlabeled_indices[selected_indices]
        x_labeled = np.concatenate((x_labeled, x_unlabeled[selected_indices]), axis=0)
        y_labeled = np.concatenate((y_labeled, y_unlabeled[selected_indices]), axis=0)
        unlabeled_indices = np.setdiff1d(unlabeled_indices, new_labeled_indices)
        x_unlabeled, y_unlabeled = x_train[unlabeled_indices], y_train[unlabeled_indices]

        model = create_cnn_model()
        model.fit(x_labeled, y_labeled, epochs=epochs_per_round, verbose=0)

        test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
        l2 = np.sqrt(sum(tf.reduce_sum(tf.square(w)).numpy() for w in model.trainable_weights))

        accuracies[approach].append(test_acc)
        l2_norms[approach].append(l2)

        print(f"✅ Accuracy: {test_acc:.4f} | L2 Norm: {l2:.2f} | Avg Entropy: {entropies.mean():.4f} | Diversity: {avg_dist:.2f}")


# Plot
plt.figure(figsize=(10, 6))
for approach in approaches:
    iterations = len(accuracies[approach])
    plt.plot(range(1, iterations + 1), accuracies[approach], marker='o', label=approach)
plt.xlabel('Active Learning Iteration')
plt.ylabel('Test Accuracy')
plt.title('Comparative Performance of Active Learning Strategies')
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

# Plot L2 norm progression
plt.figure(figsize=(10, 4))
for approach in approaches:
    iterations = len(l2_norms[approach])
    plt.plot(range(1, iterations + 1), l2_norms[approach], marker='o', label=approach)
plt.xlabel('Active Learning Iteration')
plt.ylabel('L2 Norm of Weights')
plt.title('L2 Norm Across Iterations')
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()


# Plotting
iterations = list(range(1, num_iterations + 1))
for approach in approaches:
    plt.figure(figsize=(10, 6))
    plt.plot(iterations, accuracies[approach], label='Accuracy')
    plt.plot(iterations, l2_norms[approach], label='L2 Norm')
    plt.plot(iterations, query_entropies[approach], label='Avg Entropy')
    plt.plot(iterations, query_diversity[approach], label='Diversity')
    plt.title(f'Metrics for {approach}')
    plt.xlabel('Iteration')
    plt.legend()
    plt.grid()
    plt.tight_layout()
    plt.show()

    df = pd.DataFrame({
        'Accuracy': accuracies[approach],
        'L2 Norm': l2_norms[approach],
        'Entropy': query_entropies[approach],
        'Diversity': query_diversity[approach]
    })
    sns.pairplot(df, kind='reg')
    plt.suptitle(f'Metric Relationships - {approach}')
    plt.tight_layout()
    plt.show()



2025-05-17 08:46:59.531183: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-17 08:46:59.793308: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-05-17 08:47:01.062205: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2025-05-17 08:47:03.388125: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-05-17 08:47:03.412219: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1960] Can

❌ No GPU available.

🚀 Starting approach: Random Sampling

🔁 Iteration 1


KeyboardInterrupt: 

In [1]:
import torch

if torch.cuda.is_available():
    print(f"GPU is available: {torch.cuda.get_device_name(0)}")
else:
    print("GPU is not available")


GPU is available: NVIDIA TITAN X (Pascal)
